# Hooks and tracing

You have been using hooks since the first notebook. `ReturnType` is a hook. So are `ValidateReturn`, `BudgetPool`, `RecursionLimit` and `PrintLogger`.

That is the whole extension model: a hook **observes** the run through *events*, and **influences** it by returning *effects*. Once you can write one, everything JAZ does to an agent is something you can do too.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## You have already used five of them

In [4]:
from jaz.hooks import (
    Hook,
    ReturnType,
    ValidateReturn,
    BudgetPool,
    RecursionLimit,
    IterationLimit,
    PrintLogger,
)

for cls in (ReturnType, ValidateReturn, BudgetPool, RecursionLimit, IterationLimit, PrintLogger):
    print(f"{cls.__name__:<18} is a Hook: {issubclass(cls, Hook)}")

ReturnType         is a Hook: True
ValidateReturn     is a Hook: True
BudgetPool         is a Hook: True
RecursionLimit     is a Hook: True
IterationLimit     is a Hook: True
PrintLogger        is a Hook: True


Which also explains the two ways you have been passing them.

- **Positionally** — `invoke(ReturnType(int), ...)` — applies to that one call.
- **As a context manager** — `with BudgetPool(...):` — applies to everything inside, nested invokes included.

Same object, two scopes, exactly like the difference between an input and a `scope` binding in the last notebook.

## Observing: events

A hook handles an event by defining a method named after it. `LLMQueryEnter` becomes `on_llm_query_enter`, `REPLExecExit` becomes `on_repl_exec_exit`, and so on — there is no registration step.

Return an empty list and you have a pure observer:

In [5]:
from collections import Counter

from jaz import invoke


class CountEvents(Hook):
    """Observes only. It emits no effects, so it cannot change the run."""

    def __init__(self):
        self.counts = Counter()

    # A handler is named after the event it wants: on_<event_name_in_snake_case>.
    def on_llm_query_enter(self, event):
        self.counts["llm_query_enter"] += 1
        return []

    def on_repl_exec_exit(self, event):
        self.counts["repl_exec_exit"] += 1
        return []

    def on_invoke_exit(self, event):
        self.counts["invoke_exit"] += 1
        return []


counter = CountEvents()
with counter:
    invoke(
        ReturnType(int),
        task="Print 1, then on a later turn print 2, then return 3.",
    )

for name, n in counter.counts.items():
    print(f"{name:<18} {n}")

llm_query_enter    3
repl_exec_exit     3
invoke_exit        1


Those counts are the REPL loop, measured: the agent was queried, it ran code, and that repeated until it returned.

## Influencing: effects

A handler returns a list of **effects** — changes for JAZ to apply. `AddMessages` adds to the conversation; others supply a result, modify one, drop inputs, or abort the run.

The hook below fires from the second turn onwards, and the task guarantees a second turn, so you can watch it happen rather than take it on faith:

In [6]:
from jaz.hooks.effects import AddMessages


class NudgeAfterFirstTurn(Hook):
    """Injects a message from the second turn onwards.

    `event.iteration` counts REPL turns from 0, so this fires on turn 1 and after —
    which the task below guarantees we reach.
    """

    def __init__(self):
        self.fired = 0

    def on_llm_query_enter(self, event):
        if event.iteration < 1:
            return []
        self.fired += 1
        print(f"  hook fired on turn {event.iteration}")
        return [
            AddMessages([
                {
                    "role": "user",
                    "content": "Reminder: return an int as soon as you have the answer.",
                }
            ])
        ]


nudge = NudgeAfterFirstTurn()
with nudge:
    result = invoke(
        ReturnType(int),
        task="Print 1, then on a later turn print 2, then return 3.",
    )

print(f"fired {nudge.fired} time(s); result {result}")

  hook fired on turn 1


  hook fired on turn 2


fired 2 time(s); result 3


## Tracing

`LangfuseTracing` turns those same events into spans and ships them to [Langfuse](https://langfuse.com). It needs the optional extra and an account, so it is shown rather than run here:

In [7]:
# Requires the tracing extra and a Langfuse account:
#   %pip install -q "jaz-lang[tracing]"
#
# from jaz.hooks import LangfuseTracing
#
# with LangfuseTracing(public_key=..., secret_key=..., host=...):
#     invoke(ReturnType(int), task="...")
#
# Every event above becomes a span, so a recursive run arrives as a tree you can walk.
print("tracing is optional; nothing to run here")

tracing is optional; nothing to run here


## What to take away

- **Hooks are the extension model.** Return types, validation, budgets, limits and logging are all the same mechanism.
- **Events in, effects out.** Define `on_<event>`, return a list — empty to observe, non-empty to influence.
- **Positional is local; `with` propagates.**

---

## Where to go next

- [Concepts](https://jaz-lang.ai/docs/getting-started/concepts) — the same ideas, written down.
- [Hooks reference](https://jaz-lang.ai/docs/reference/hooks) — the full event and effect vocabulary.
- [Install locally](https://jaz-lang.ai/docs/getting-started/installation) — when you want this in a real project.